# 04 — Excel and CSV Processing

**FinRAG learning series · Notebook 4 of 14**

## Learning objectives
1. Read Excel workbooks with **openpyxl**: sheets, cell coordinates, values and **formulas**.
2. Understand cached formula values and why FinRAG never evaluates formulas itself.
3. Read legacy `.xls` and CSV files (encoding and delimiter detection).
4. Use FinRAG's tabular logic: header detection, wide vs record ("long") layouts, Python-computed totals.
5. Keep **sheet + cell** (Excel) and **row** (CSV) references for every value.

## Prerequisites
Notebooks 01 and 03. Saved outputs use `.xlsx`, `.xls` and `.csv` files transcribed from a public report's statement page
(see notebook 03); one experiment adds a formula to a *copy* of the workbook — clearly labelled.

## Concept: a spreadsheet is a grid of typed cells
Each cell has a coordinate (`B9`), a value (number, text, date) and possibly a **formula** (`=B7+B8`). An `.xlsx` stores both the
formula and the **cached result** of the last calculation in Excel. Files written by other tools often have formulas *without*
cached results.

FinRAG loads every workbook twice: `data_only=False` (formulas) and `data_only=True` (cached values). If a formula has no cached
value it is **reported and left empty** — evaluating formulas would mean re-implementing Excel, and guessing is not allowed.

### FinRAG's spreadsheet workflow
```
sheet grid ─► first header-like row (text cells followed by numbers)  ─► rows above = titles/unit lines (context)
           ─► wide layout  : label column(s) + value columns (periods, Budget/Actual…)
           ─► long layout  : >30 records with non-period numeric columns → also compute totals & per-category sums in Python
           ─► each value: Location(sheet, cell, row), column header kept, formula kept
```

In [1]:
# Setup: make the production backend importable and keep secrets out of outputs
import sys, os
sys.path.insert(0, os.path.abspath("."))      # run notebooks from the notebooks/ folder
import finrag_nb as nb                          # helper: paths, safe settings, DB session
print("Backend package:", nb.BACKEND_DIR)
print("Documents folder:", nb.DOCS_DIR)

Backend package: /home/user/Ready_To_Deploy/backend
Documents folder: /opt/finrag_testdocs


## 1. Sheets, cells and coordinates with openpyxl

In [2]:
import openpyxl
xlsx_path = nb.doc_path("derived/derived_p58.xlsx")
wb_values = openpyxl.load_workbook(xlsx_path, data_only=True)
print("sheets:", wb_values.sheetnames)
ws = wb_values["Statement"]
print("used range:", ws.dimensions)
for row in ws.iter_rows(min_row=1, max_row=8):
    print([(c.coordinate, c.value) for c in row if c.value is not None])

sheets: ['Statement', 'Provenance']
used range: A1:C32
[('A1', 'Statement of Profit and Loss')]
[('A2', '(C in crore)')]
[('A3', '(C in crore)')]
[('A4', 'Particulars'), ('B4', '2024-25'), ('C4', '2023-24')]
[('A5', 'Value of Sales'), ('B5', '5,38,564'), ('C5', '5,33,566')]
[('A6', 'Income from Services'), ('B6', '18,599'), ('C6', '41,390')]
[('A7', 'Value of Sales & Services (Revenue)'), ('B7', '5,57,163'), ('C7', '5,74,956')]
[('A8', 'Less: GST Recovered'), ('B8', '24,371'), ('C8', '27,014')]


## 2. Experiment: formulas and cached values
The transcribed workbook contains only values. To see how formulas behave we add one formula to a **temporary copy** that sums two
real cells (Revenue from Operations + Other Income for 2024-25). This is an experiment, not data: openpyxl writes the formula but no
cached result, exactly like many exported files.

In [3]:
import shutil, tempfile
from decimal import Decimal
from app.financial_analytics.numbers import parse_number

tmpdir = tempfile.mkdtemp()
copy = f"{tmpdir}/experiment_copy.xlsx"
shutil.copy(xlsx_path, copy)
wbf = openpyxl.load_workbook(copy)
wsf = wbf["Statement"]
labels = {wsf.cell(r, 1).value: r for r in range(1, wsf.max_row + 1)}
r_rev, r_oth = labels["Revenue from Operations"], labels["Other Income"]
wsf.cell(wsf.max_row + 2, 1, "EXPERIMENT: Revenue + Other income (formula)")
wsf.cell(wsf.max_row, 2, f"=B{r_rev}+B{r_oth}")
wbf.save(copy)
print("formula written:", wsf.cell(wsf.max_row, 2).value, "in cell", wsf.cell(wsf.max_row, 2).coordinate)

f = openpyxl.load_workbook(copy, data_only=False)["Statement"]
v = openpyxl.load_workbook(copy, data_only=True)["Statement"]
cell = f"B{f.max_row}"
print("data_only=False ->", f[cell].value)
print("data_only=True  ->", v[cell].value, "(no cached value: the file was never recalculated in Excel)")
print("Note: the transcribed cells are text, so Excel itself would need =VALUE(...) — another reason not to evaluate formulas blindly.")

formula written: =B9+B10 in cell B34
data_only=False -> =B9+B10
data_only=True  -> None (no cached value: the file was never recalculated in Excel)
Note: the transcribed cells are text, so Excel itself would need =VALUE(...) — another reason not to evaluate formulas blindly.


In [4]:
from app.document_processing.parsers.excel_parser import XlsxParser
parsed_copy = XlsxParser().parse(copy)
print("warnings reported by FinRAG:")
for w in parsed_copy.warnings:
    print(" -", w)

warnings reported by FinRAG:
 - Sheet 'Statement': 1 formula cell(s) have no cached value (file not recalculated in Excel); those cells were not evaluated and are treated as empty.


## 3. The production Excel parser on the original workbook

In [5]:
from collections import Counter
parsed = XlsxParser().parse(str(xlsx_path))
print("sheets:", parsed.sheet_names, "| elements:", Counter(e.kind for e in parsed.elements), "| warnings:", parsed.warnings)
for e in [e for e in parsed.elements if e.kind == "table_row"][:5]:
    print(f"{e.location.describe():<28} | {nb.shorten(e.text, 70)}")
    for c in e.cells:
        print(f"      value {c.text!r:>12} at {c.location.describe()} (column header: {c.column_header!r})")

sheets: ['Statement', 'Provenance'] | elements: Counter({'table_row': 28, 'heading': 4}) | warnings: []
sheet 'Statement' cells A5:C5 | Value of Sales | 2024-25: 5,38,564 | 2023-24: 5,33,566
      value   '5,38,564' at sheet 'Statement' cell B5 (column header: '2024-25')
      value   '5,33,566' at sheet 'Statement' cell C5 (column header: '2023-24')
sheet 'Statement' cells A6:C6 | Income from Services | 2024-25: 18,599 | 2023-24: 41,390
      value     '18,599' at sheet 'Statement' cell B6 (column header: '2024-25')
      value     '41,390' at sheet 'Statement' cell C6 (column header: '2023-24')
sheet 'Statement' cells A7:C7 | Value of Sales & Services (Revenue) | 2024-25: 5,57,163 | 2023-24: 5,…
      value   '5,57,163' at sheet 'Statement' cell B7 (column header: '2024-25')
      value   '5,74,956' at sheet 'Statement' cell C7 (column header: '2023-24')
sheet 'Statement' cells A8:C8 | Less: GST Recovered | 2024-25: 24,371 | 2023-24: 27,014
      value     '24,371' at sheet 'Statemen

## 4. Legacy .xls

In [6]:
from app.document_processing.parsers.excel_parser import XlsParser
xls = XlsParser().parse(str(nb.doc_path("derived/derived_p58.xls")))
print("warnings:", xls.warnings)
r = [e for e in xls.elements if e.kind == "table_row"][4]
print(r.location.describe(), "|", r.text)

warnings: ['Legacy .xls format: formulas cannot be read, only stored values.']
sheet 'Statement' cells A9:C9 | Revenue from Operations | 2024-25: 5,32,792 | 2023-24: 5,47,942


## 5. CSV: encoding and delimiter detection
CSV has no types, no sheets and no standard delimiter. FinRAG tries UTF-8 (with/without BOM), cp1252 and latin-1, sniffs the
delimiter, and reads every cell as a string.

In [7]:
from app.document_processing.parsers.csv_parser import CsvParser
csv_path = nb.doc_path("derived/derived_p58.csv")
c = CsvParser().parse(str(csv_path))
print("file metadata:", c.file_metadata, "| rows:", c.row_count)
for e in [e for e in c.elements if e.kind == "table_row"][:4]:
    print(f"{e.location.describe():<8} | {e.text}")

file metadata: {'encoding': 'utf-8-sig', 'delimiter': ','} | rows: 29
row 2    | Value of Sales | 2024-25: 5,38,564 | 2023-24: 5,33,566
row 3    | Income from Services | 2024-25: 18,599 | 2023-24: 41,390
row 4    | Value of Sales & Services (Revenue) | 2024-25: 5,57,163 | 2023-24: 5,74,956
row 5    | Less: GST Recovered | 2024-25: 24,371 | 2023-24: 27,014


## 6. Record (long) layouts and Python-computed summaries
Transaction lists, expense statements and budgets are often *records*: one row per item with a category and an amount. Asking an LLM
"which category is highest?" over hundreds of rows is unreliable, so FinRAG computes totals and per-category sums **in Python** and
indexes them as clearly labelled `[Computed by FinRAG]` text.

To see this on real numbers we reshape the transcribed statement into records (one row per line item and period) — the values are
unchanged, only the layout differs.

In [8]:
import pandas as pd
wide = pd.read_csv(csv_path, dtype=str)
records = wide.melt(id_vars="Particulars", var_name="Period", value_name="Amount")
rec_csv = f"{tmpdir}/records_layout.csv"
records.to_csv(rec_csv, index=False)
print(len(records), "records written (same values, record layout)")

rp = CsvParser().parse(rec_csv)
for e in rp.elements:
    if "computed_summary" in e.context:
        print("\n" + nb.shorten(e.text, 400))

56 records written (same values, record layout)

[Computed by FinRAG from 56 rows] Column 'Amount': total 6827371.26, minimum -10679, maximum 574956, rows with values 56 of 56.

[Computed by FinRAG] Sum of 'Amount' grouped by 'Particulars' (highest first): Value of Sales & Services (Revenue): 1132119 (2 rows); Total Income: 1108956 (2 rows); Revenue from Operations: 1080734 (2 rows); Value of Sales: 1072130 (2 rows); Total Expenses: 1007555 (2 rows); Cost of Materials Consumed: 760439 (2 rows); Other Expenses: 121275 (2 rows); Profit Before Tax: 101401 (2 rows); Profit f…

[Computed by FinRAG] Sum of 'Amount' grouped by 'Period' (highest first): 2023-24: 3459439.14 (28 rows); 2024-25: 3367932.12 (28 rows).


## Interpretation
* Every Excel value carries its sheet and cell (e.g. *sheet 'Statement' cell B9*); CSV values carry their row number.
* The formula experiment showed a formula **without a cached value**; FinRAG reported it and did not evaluate it.
* The summaries are **arithmetically exact but financially meaningless here**: a statement of profit and loss contains subtotals
  (*Total Income* = its components) and per-share figures (EPS), so adding all rows double-counts and mixes units — notice the
  decimal total caused by EPS. This is the important lesson: code guarantees correct arithmetic, not correct *meaning*. FinRAG
  therefore computes record summaries only for genuine record layouts (ledgers, transaction lists, budgets), labels them as
  computed, and the evidence shows exactly which rows were summed.

## Common errors and troubleshooting
| Symptom | Cause | Fix |
|---|---|---|
| Formula cells empty | No cached values | Open and save the file in Excel/LibreOffice to recalculate |
| `.xlsm` rejected | Macro-enabled | Save as `.xlsx` |
| Wrong header row | Title block above the table is unusual | Check `_find_header` in `tabular.py`; put headers in one row |
| CSV columns merged | Unusual delimiter | The sniffer supports `, ; tab |`; others need conversion |
| Dates shown as numbers in .xls | xlrd returns serial numbers for some formats | Prefer .xlsx |

## Practical exercises
1. Upload your own budget spreadsheet with *Budget* and *Actual* columns. Which values are line items and which canonical metrics?
2. Create an expense ledger CSV from your real records (≥31 rows: Date, Category, Amount) and look at the computed category sums.
3. Open a workbook with formulas in Excel, save it, and parse it again — do the uncached-formula warnings disappear?

## Summary
You read workbooks at cell level, learnt why cached formula values matter, parsed .xlsx/.xls/.csv with the production parsers, and saw
how record layouts get Python-computed summaries with traceable row ranges.

## Further learning
* openpyxl: https://openpyxl.readthedocs.io/
* pandas IO tools: https://pandas.pydata.org/docs/user_guide/io.html
* FinRAG source: `parsers/excel_parser.py`, `parsers/csv_parser.py`, `parsers/tabular.py`
* Next: **05 — OCR experiments**.